# 🤖 Research Paper AI Assistant — All-in-One Notebook

## 📌 Overview
This single notebook contains the **complete AI pipeline** for the Research Paper Assistant system.
It runs a **FastAPI server on `http://localhost:8000`** that your MERN web application connects to.

### What this notebook does:
```
PDF Upload  →  Text Extraction  →  NLP Preprocessing  →  Chunking
     →  Embeddings (MiniLM)  →  ChromaDB Storage
     →  Retrieval  →  Answer Generation (Flan-T5)  →  FastAPI Server
```

### API Endpoints (available after running the last cell):
| Method | Endpoint | Description |
|--------|----------|-------------|
| `GET`  | `http://localhost:8000/health` | Check if API is running |
| `POST` | `http://localhost:8000/upload` | Upload a PDF file |
| `POST` | `http://localhost:8000/chat` | Ask a question |
| `POST` | `http://localhost:8000/summarize` | Summarize the paper |
| `POST` | `http://localhost:8000/keypoints` | Extract key points |
| `GET`  | `http://localhost:8000/docs` | Swagger UI (test endpoints in browser) |

---

## ✅ Before You Start — Prerequisites

Make sure you have these installed on your machine:
- **Python 3.9 or higher** → check with `python --version` in terminal
- **VS Code** with the **Jupyter extension** installed
- A Python **virtual environment** is strongly recommended (see Step 1)

## 📁 Recommended Project Folder Structure
```
your-project/
├── research_paper_ai.ipynb   ← this file
├── chroma_db/                ← auto-created when you upload a PDF
├── mern-app/
│   ├── client/               ← React frontend
│   ├── server/               ← Express.js backend
│   └── .env                  ← contains PYTHON_API_URL=http://localhost:8000
```

> ⚠️ **Important:** Keep this notebook **running** while using the MERN app.
> The last cell starts the server — do not interrupt it during use.

---
## 📦 STEP 1 — Install All Required Libraries

This cell installs every Python library needed for the entire pipeline:

| Library | Purpose |
|---|---|
| `PyMuPDF` | Extract text from PDF files |
| `nltk` | Sentence tokenization and NLP tools |
| `sentence-transformers` | Convert text chunks into vector embeddings |
| `chromadb` | Vector database to store and search embeddings |
| `transformers` | Load and run the Flan-T5 language model |
| `torch` | Deep learning backend for running AI models |
| `fastapi` | Build the web API server |
| `uvicorn` | ASGI server to run FastAPI |
| `python-multipart` | Enables file upload in FastAPI |
| `nest-asyncio` | Allows running async server inside a notebook |

> ⏱️ **First run only:** This may take 3–5 minutes. Subsequent runs are instant.
>
> 💡 **VS Code Tip:** Select your Python interpreter first — press `Ctrl+Shift+P` → "Python: Select Interpreter"

In [1]:
import sys

print('📦 Installing required libraries...')
print('   This may take a few minutes on first run.\n')

!{sys.executable} -m pip install --quiet \
    PyMuPDF \
    nltk \
    sentence-transformers \
    chromadb \
    transformers \
    torch \
    fastapi \
    uvicorn \
    python-multipart \
    nest-asyncio

print('\n✅ All libraries installed successfully!')

📦 Installing required libraries...
   This may take a few minutes on first run.


✅ All libraries installed successfully!


---
## 📥 STEP 2 — Download NLTK Language Data

NLTK (Natural Language Toolkit) needs some extra data files to work:
- **`punkt`** and **`punkt_tab`** — used to split text into individual sentences
- **`stopwords`** — a list of common words like 'the', 'is', 'at' (used optionally)

> These are downloaded to your home folder (`~/nltk_data`) and only need to be downloaded once.

In [2]:
import nltk

print('📥 Downloading NLTK language data...')
nltk.download('punkt',      quiet=True)
nltk.download('punkt_tab',  quiet=True)
nltk.download('stopwords',  quiet=True)

print('✅ NLTK data ready!')

📥 Downloading NLTK language data...
✅ NLTK data ready!


---
## 🤖 STEP 3 — Load AI Models

We load two AI models that power the assistant:

### Model 1: `all-MiniLM-L6-v2` (Sentence Transformer)
- Converts text into **384-dimensional vectors** (embeddings)
- Used to find which chunks of your paper are most relevant to a question
- Size: ~90 MB — downloads once and is cached locally

### Model 2: `google/flan-t5-base` (Language Model)
- A free, instruction-following language model by Google
- Reads the retrieved context + your question and **generates a natural answer**
- Size: ~990 MB — downloads once and is cached at `~/.cache/huggingface/`

> ⏱️ **First run:** Model downloads may take 5–10 minutes depending on internet speed.
> **Subsequent runs:** Models load from local cache in ~30 seconds.
>
> 💡 If you have an **NVIDIA GPU**, PyTorch will use it automatically for faster performance.

In [3]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Detect if a GPU is available, otherwise use CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Running on: {device.upper()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
else:
    print('   (No GPU found — CPU mode is fine, just a bit slower)')

# ── Load Sentence Transformer (Embedding Model) ──────────────
print('\n📥 Loading embedding model (all-MiniLM-L6-v2)...')
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Embedding model ready!')

# ── Load Flan-T5 (Answer Generation Model) ───────────────────
print('\n📥 Loading Flan-T5 language model (google/flan-t5-base)...')
print('   First run downloads ~990 MB — please wait...')
tokenizer  = T5Tokenizer.from_pretrained('google/flan-t5-base')
gen_model  = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base')
gen_model  = gen_model.to(device)
gen_model.eval()   # Set to inference mode (not training)
print('✅ Flan-T5 model ready!')

print('\n🎉 All AI models loaded successfully!')

C:\Users\lahir\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🖥️  Running on: CPU
   (No GPU found — CPU mode is fine, just a bit slower)

📥 Loading embedding model (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4012.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model ready!

📥 Loading Flan-T5 language model (google/flan-t5-base)...
   First run downloads ~990 MB — please wait...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3543.90it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Flan-T5 model ready!

🎉 All AI models loaded successfully!


---
## 🗄️ STEP 4 — Initialize ChromaDB (Vector Database)

**ChromaDB** is a lightweight vector database that:
- Stores the text chunks from your PDF along with their vector embeddings
- Lets us search for the most relevant chunks when a question is asked
- Uses **cosine similarity** to measure how close two vectors are

### What gets created:
- A `chroma_db/` folder is created **in the same directory as this notebook**
- This folder persists on disk — you don't lose data if you restart the notebook
- A new paper upload will replace the previous one's data

> 📁 You will see a `chroma_db/` folder appear next to this notebook after the first PDF upload.

In [4]:
import chromadb
import os

# Get the directory where this notebook is saved
NOTEBOOK_DIR = os.getcwd()
CHROMA_PATH  = os.path.join(NOTEBOOK_DIR, 'chroma_db')

print(f'📁 ChromaDB will be stored at: {CHROMA_PATH}')

# Create persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Try to load an existing collection, or create a fresh one
COLLECTION_NAME = 'research_paper'
try:
    collection = chroma_client.get_collection(name=COLLECTION_NAME)
    print(f'✅ Existing ChromaDB collection loaded ({collection.count()} chunks found)')
    print('   (A previous paper is already in memory — upload a new one to replace it)')
except Exception:
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={'hnsw:space': 'cosine'}
    )
    print('✅ Fresh ChromaDB collection created (ready for first PDF upload)')

📁 ChromaDB will be stored at: c:\Users\lahir\Downloads\claude 2 research paper\chroma_db
✅ Fresh ChromaDB collection created (ready for first PDF upload)


---
## 🛠️ STEP 5 — Define the Full AI Pipeline

This cell defines all the helper functions that power the API endpoints.
Think of this as the "engine" of the system.

### Functions defined here:

| Function | What it does |
|---|---|
| `extract_text(pdf_bytes)` | Reads a PDF file and returns all its text |
| `clean_text(text)` | Removes noise, special characters, URLs |
| `create_chunks(sentences)` | Splits text into overlapping chunks of ~500 characters |
| `store_chunks(chunks)` | Embeds all chunks and saves them to ChromaDB |
| `retrieve_context(query)` | Finds the most relevant chunks for a question |
| `generate_response(prompt)` | Passes a prompt to Flan-T5 and returns the answer |

### Why overlapping chunks?
```
Chunk 1:  [Sentence A] [Sentence B] [Sentence C]
Chunk 2:              [Sentence B] [Sentence C] [Sentence D] [Sentence E]
```
Overlapping ensures no important context is cut off at chunk boundaries.

In [5]:
import re
import fitz   # PyMuPDF
from nltk.tokenize import sent_tokenize


# ── 1. PDF Text Extraction ────────────────────────────────────
def extract_text(pdf_bytes: bytes) -> str:
    """
    Opens a PDF from raw bytes and extracts all text page by page.

    Args:
        pdf_bytes: Raw bytes of the uploaded PDF file

    Returns:
        Full extracted text as a single string
    """
    doc = fitz.open(stream=pdf_bytes, filetype='pdf')
    full_text = ''
    for page in doc:
        full_text += page.get_text() + '\n'
    doc.close()
    return full_text


# ── 2. Text Cleaning ──────────────────────────────────────────
def clean_text(text: str) -> str:
    """
    Removes noise from raw PDF text:
    - Page break characters (\x0c)
    - Tab characters
    - Non-ASCII unicode symbols
    - URLs and email addresses
    - Excessive whitespace and blank lines

    Args:
        text: Raw text from PDF

    Returns:
        Cleaned and normalized text
    """
    text = text.replace('\x0c', ' ')            # Remove page breaks
    text = text.replace('\t', ' ')              # Remove tabs
    text = re.sub(r'[^\x00-\x7F]+', ' ', text) # Remove non-ASCII characters
    text = re.sub(r'http\S+|www\.\S+', '', text)# Remove URLs
    text = re.sub(r'\S+@\S+', '', text)         # Remove email addresses
    text = re.sub(r'\n{2,}', '\n', text)        # Collapse multiple newlines
    text = re.sub(r' {2,}', ' ', text)          # Collapse multiple spaces
    return text.strip()


# ── 3. Text Chunking ──────────────────────────────────────────
def create_chunks(sentences: list, chunk_size: int = 500, overlap: int = 100) -> list:
    """
    Groups sentences into overlapping text chunks.

    Why chunks? AI models have a limit on input length.
    We split the paper into small overlapping pieces so the model
    can focus on the most relevant section for each question.

    Args:
        sentences:  List of individual sentences
        chunk_size: Target max characters per chunk (default 500)
        overlap:    Characters to share between consecutive chunks (default 100)

    Returns:
        List of text chunk strings
    """
    chunks  = []
    current = ''

    for sentence in sentences:
        if len(current) + len(sentence) <= chunk_size:
            current += ' ' + sentence
        else:
            if current.strip():
                chunks.append(current.strip())
            # Start next chunk with overlap from previous
            current = current[-overlap:] + ' ' + sentence

    if current.strip():
        chunks.append(current.strip())

    return chunks


# ── 4. Embed + Store in ChromaDB ──────────────────────────────
def store_chunks(chunks: list):
    """
    Converts text chunks to vector embeddings and stores them in ChromaDB.

    Process:
    1. Delete old collection (replacing previous paper)
    2. Create fresh collection
    3. Generate embeddings for all chunks using MiniLM model
    4. Store text + embeddings in ChromaDB

    Args:
        chunks: List of text chunks to store
    """
    global collection

    # Clear previous paper data
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    # Create fresh collection
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={'hnsw:space': 'cosine'}
    )

    # Generate embeddings for all chunks at once (batched for efficiency)
    embeddings = embedding_model.encode(
        chunks,
        batch_size=32,
        show_progress_bar=False,
        convert_to_numpy=True
    )

    # Store in ChromaDB
    collection.add(
        ids        = [str(i) for i in range(len(chunks))],
        documents  = chunks,
        embeddings = embeddings.tolist()
    )


# ── 5. Retrieve Relevant Context ──────────────────────────────
def retrieve_context(query: str, top_k: int = 3) -> str:
    """
    Finds the most relevant text chunks for a given question.

    Process:
    1. Convert the question to a vector embedding
    2. Search ChromaDB for the top_k most similar chunk vectors
    3. Return those chunks combined as a single context string

    Args:
        query:  The user's question
        top_k:  Number of chunks to retrieve (default 3)

    Returns:
        Combined context string from top matching chunks
    """
    query_embedding = embedding_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, collection.count())
    )
    return ' '.join(results['documents'][0])


# ── 6. Generate AI Response ───────────────────────────────────
def generate_response(prompt: str, max_tokens: int = 256) -> str:
    """
    Passes a prompt to the Flan-T5 model and returns the generated text.

    Process:
    1. Tokenize the prompt (convert text to numbers)
    2. Run the model to generate output tokens
    3. Decode the output back to readable text

    Args:
        prompt:     Full instruction + context + question
        max_tokens: Maximum length of the generated answer

    Returns:
        Generated answer as a string
    """
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        max_length=1024,
        truncation=True
    ).to(device)

    with torch.no_grad():   # Disable gradient tracking for faster inference
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens    = max_tokens,
            num_beams         = 4,     # Beam search for better quality
            early_stopping    = True,
            no_repeat_ngram_size = 3   # Prevent repetitive output
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


print('✅ All pipeline functions defined and ready!')

✅ All pipeline functions defined and ready!


---
## 🌐 STEP 6 — Define FastAPI Application & All Endpoints

Here we build the actual web API using **FastAPI**.

### CORS (Cross-Origin Resource Sharing)
Your MERN app runs on `http://localhost:3000` (React) and `http://localhost:5000` (Express).
These are different "origins" from `http://localhost:8000` (this API).
We enable CORS so browsers allow these cross-origin requests.

### Endpoints explained:

**`GET /health`**
- Simple ping — your MERN app can call this on startup to verify the AI API is running

**`POST /upload`**
- Accepts a PDF file from a form upload
- Runs the full pipeline: extract → clean → chunk → embed → store
- Returns stats about the processed paper

**`POST /chat`**
- Accepts `{ "question": "..." }` as JSON body
- Retrieves relevant chunks → builds prompt → generates answer
- Returns `{ "question": "...", "answer": "..." }`

**`POST /summarize`**
- No body needed — works on the currently uploaded paper
- Returns a 3–5 sentence summary

**`POST /keypoints`**
- No body needed — works on the currently uploaded paper
- Returns the 3 most important contributions or findings

In [6]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel


# ── Create FastAPI App ────────────────────────────────────────
app = FastAPI(
    title       = 'Research Paper AI API',
    description = 'RAG-based API for interacting with research papers. Connect your MERN app to this.',
    version     = '1.0.0'
)


# ── CORS Middleware ───────────────────────────────────────────
# Allows your React (port 3000) and Express (port 5000) apps to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins  = [
        'http://localhost:3000',   # React frontend
        'http://localhost:5000',   # Express backend
        'http://127.0.0.1:3000',
        'http://127.0.0.1:5000',
    ],
    allow_methods  = ['*'],
    allow_headers  = ['*'],
)


# ── Request/Response Schemas ──────────────────────────────────
class ChatRequest(BaseModel):
    question: str   # The user's question, e.g. { "question": "What is the methodology?" }


# ── GET /health ───────────────────────────────────────────────
@app.get('/health')
def health_check():
    """
    Health check endpoint.
    Your MERN app can call this on startup to confirm the Python API is running.

    Returns:
        status, message, and number of chunks currently loaded
    """
    return {
        'status'        : 'ok',
        'message'       : 'Research Paper AI API is running on port 8000',
        'chunks_loaded' : collection.count(),
        'paper_ready'   : collection.count() > 0
    }


# ── POST /upload ──────────────────────────────────────────────
@app.post('/upload')
async def upload_paper(file: UploadFile = File(...)):
    """
    Upload a research paper PDF.

    This endpoint:
    1. Reads the uploaded PDF bytes
    2. Extracts and cleans all text
    3. Splits into overlapping chunks
    4. Generates embeddings for each chunk
    5. Stores everything in ChromaDB

    MERN usage example (Express.js with multer or axios FormData):
        const formData = new FormData();
        formData.append('file', pdfFile);
        await axios.post('http://localhost:8000/upload', formData);
    """
    # Validate file type
    if not file.filename.lower().endswith('.pdf'):
        raise HTTPException(status_code=400, detail='Only PDF files are supported')

    try:
        # Read uploaded file into memory
        pdf_bytes = await file.read()

        # Step 1: Extract raw text from PDF
        raw_text = extract_text(pdf_bytes)
        if not raw_text.strip():
            raise HTTPException(status_code=422, detail='Could not extract text. This may be a scanned/image PDF.')

        # Step 2: Clean the text
        cleaned_text = clean_text(raw_text)

        # Step 3: Split into sentences then create overlapping chunks
        sentences = sent_tokenize(cleaned_text)
        chunks    = create_chunks(sentences)

        # Step 4: Generate embeddings and store in ChromaDB
        store_chunks(chunks)

        return {
            'status'           : 'success',
            'filename'         : file.filename,
            'total_chunks'     : len(chunks),
            'total_characters' : len(cleaned_text),
            'total_sentences'  : len(sentences),
            'message'          : 'Paper processed successfully! You can now ask questions.'
        }

    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'Processing error: {str(e)}')


# ── POST /chat ────────────────────────────────────────────────
@app.post('/chat')
def chat(request: ChatRequest):
    """
    Ask a question about the uploaded research paper.

    This endpoint:
    1. Converts the question to a vector embedding
    2. Finds the 3 most relevant chunks from ChromaDB
    3. Builds a prompt with the context + question
    4. Generates an answer using Flan-T5

    MERN usage example (Express.js or React):
        await axios.post('http://localhost:8000/chat', { question: userMessage });

    Request body:  { "question": "What is the proposed methodology?" }
    Response body: { "question": "...", "answer": "..." }
    """
    if collection.count() == 0:
        raise HTTPException(
            status_code=400,
            detail='No paper uploaded yet. Please call /upload first.'
        )

    if not request.question.strip():
        raise HTTPException(status_code=400, detail='Question cannot be empty.')

    try:
        # Retrieve relevant context from ChromaDB
        context = retrieve_context(request.question, top_k=3)

        # Build prompt for Flan-T5
        prompt = (
            f'Answer the question based on the context below.\n\n'
            f'Context: {context[:1500]}\n\n'
            f'Question: {request.question}\n\n'
            f'Answer:'
        )

        answer = generate_response(prompt, max_tokens=256)

        return {
            'question' : request.question,
            'answer'   : answer
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ── POST /summarize ───────────────────────────────────────────
@app.post('/summarize')
def summarize():
    """
    Generate a concise summary of the uploaded research paper.

    Uses broad retrieval terms to collect the most informative sections
    (abstract, introduction, conclusion, methodology), then asks Flan-T5
    to summarize them in 3–5 sentences.

    MERN usage example:
        await axios.post('http://localhost:8000/summarize');

    Response body: { "summary": "..." }
    """
    if collection.count() == 0:
        raise HTTPException(status_code=400, detail='No paper uploaded yet.')

    try:
        context = retrieve_context(
            'abstract introduction conclusion methodology results contribution',
            top_k=5
        )

        prompt = (
            f'Summarize the following research paper content in 3 to 5 sentences. '
            f'Focus on the main problem, approach, and key findings.\n\n'
            f'Content: {context[:1500]}\n\n'
            f'Summary:'
        )

        summary = generate_response(prompt, max_tokens=300)
        return {'summary': summary}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ── POST /keypoints ───────────────────────────────────────────
@app.post('/keypoints')
def key_points():
    """
    Extract the 3 most important contributions or findings from the paper.

    MERN usage example:
        await axios.post('http://localhost:8000/keypoints');

    Response body: { "key_points": "..." }
    """
    if collection.count() == 0:
        raise HTTPException(status_code=400, detail='No paper uploaded yet.')

    try:
        context = retrieve_context(
            'key contributions findings results conclusions novelty',
            top_k=5
        )

        prompt = (
            f'List the 3 most important contributions, findings, or results '
            f'from this research paper.\n\n'
            f'Content: {context[:1500]}\n\n'
            f'Key points:'
        )

        points = generate_response(prompt, max_tokens=250)
        return {'key_points': points}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print('✅ FastAPI app and all endpoints defined!')
print('\n📋 Endpoints ready:')
print('   GET  http://localhost:8000/health')
print('   POST http://localhost:8000/upload')
print('   POST http://localhost:8000/chat')
print('   POST http://localhost:8000/summarize')
print('   POST http://localhost:8000/keypoints')
print('   GET  http://localhost:8000/docs      ← Swagger UI')

✅ FastAPI app and all endpoints defined!

📋 Endpoints ready:
   GET  http://localhost:8000/health
   POST http://localhost:8000/upload
   POST http://localhost:8000/chat
   POST http://localhost:8000/summarize
   POST http://localhost:8000/keypoints
   GET  http://localhost:8000/docs      ← Swagger UI


---
## 🚀 STEP 7 — Start the API Server

This is the **final cell**. Running it starts the FastAPI server on `http://localhost:8000`.

### ⚠️ Important:
- This cell will **run forever** — that is intentional. The server must stay running while your MERN app is active.
- Do **not** stop this cell while using the web app.
- To stop the server, click the **Stop** button (■) on this cell in VS Code.

### After running this cell:
1. Open your browser and visit **`http://localhost:8000/docs`** to see the Swagger UI and test endpoints
2. Start your MERN app normally (`npm start` / `npm run dev`)
3. Your Express backend should have `PYTHON_API_URL=http://localhost:8000` in its `.env` file

### MERN `.env` setup:
```env
# In your MERN server/.env file
PYTHON_API_URL=http://localhost:8000
MONGODB_URI=your_mongodb_connection_string
PORT=5000
```

### Express.js proxy example (how your MERN backend calls this API):
```javascript
// server/routes/ai.js
const axios = require('axios');
const PYTHON_API = process.env.PYTHON_API_URL;

// Proxy upload to Python API
router.post('/upload', upload.single('file'), async (req, res) => {
  const formData = new FormData();
  formData.append('file', fs.createReadStream(req.file.path));
  const response = await axios.post(`${PYTHON_API}/upload`, formData);
  res.json(response.data);
});

// Proxy chat to Python API
router.post('/chat', async (req, res) => {
  const response = await axios.post(`${PYTHON_API}/chat`, {
    question: req.body.question
  });
  res.json(response.data);
});
```

In [9]:
import uvicorn
import asyncio

print('🚀 Starting Research Paper AI API...')
print('='*55)
print('  Server URL  :  http://localhost:8000')
print('  Swagger Docs:  http://localhost:8000/docs')
print('  Health Check:  http://localhost:8000/health')
print('='*55)
print('⚠️  Keep this cell running while using the MERN app.')
print('   Press the Stop (■) button to shut down the server.\n')

# Start the server using the existing notebook event loop
config = uvicorn.Config(
    app     = app,
    host    = '0.0.0.0',   # Accept connections from localhost and local network
    port    = 8000,
    reload  = False        # Reload not supported inside notebooks
)
server = uvicorn.Server(config)

# In modern Jupyter notebooks, you can use 'await' at the top level
await server.serve()

🚀 Starting Research Paper AI API...
  Server URL  :  http://localhost:8000
  Swagger Docs:  http://localhost:8000/docs
  Health Check:  http://localhost:8000/health
⚠️  Keep this cell running while using the MERN app.
   Press the Stop (■) button to shut down the server.



INFO:     Started server process [11280]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:4327 - "GET /health HTTP/1.1" 200 OK


C:\Users\lahir\AppData\Roaming\Python\Python313\site-packages\nltk\tabdata.py:47: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return tuple(s.split("\t"))


INFO:     127.0.0.1:6576 - "POST /upload HTTP/1.1" 200 OK
INFO:     127.0.0.1:1268 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:5966 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:5966 - "POST /summarize HTTP/1.1" 200 OK
INFO:     127.0.0.1:5966 - "POST /keypoints HTTP/1.1" 200 OK
INFO:     127.0.0.1:13181 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:6830 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:14929 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:7207 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:1615 - "POST /keypoints HTTP/1.1" 200 OK
INFO:     127.0.0.1:8656 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:4605 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:2267 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:3551 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:2778 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:6574 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:10268 - "POST /summarize HTTP/1.1" 200 OK
INFO:     127.

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [11280]
